# 文本转语音 (TTS) 高级教程

本教程深入讲解 TTS 系统的高级技术：
- 声学模型架构
- 声码器原理
- 韵律控制
- 说话人克隆

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional, Tuple

from tts import TextToSpeech, TTSConfig, create_tts_model, tts_loss

## 1. 声学模型训练

In [ ]:
class TTSTrainer:
    """TTS 训练器"""
    
    def __init__(self, model: TextToSpeech, lr: float = 1e-4):
        self.model = model
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    def train_step(self, text: torch.Tensor, mel_target: torch.Tensor,
                   stop_target: torch.Tensor, text_mask: torch.Tensor = None) -> Dict:
        """单步训练"""
        self.model.train()
        self.optimizer.zero_grad()
        
        outputs = self.model(text, mel_target, text_mask)
        
        total_loss, loss_dict = tts_loss(
            outputs['mel_output'],
            outputs['mel_postnet'],
            mel_target,
            outputs['stop_tokens'],
            stop_target
        )
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        
        return {k: v.item() for k, v in loss_dict.items()}

# 测试
model = create_tts_model("tiny")
trainer = TTSTrainer(model)

text = torch.randint(0, 256, (2, 50))
mel = torch.randn(2, 80, 200)
stop = torch.zeros(2, 200)
stop[:, -10:] = 1.0

losses = trainer.train_step(text, mel, stop)
print(f"Losses: {losses}")

## 2. 韵律控制

In [ ]:
class ProsodyController:
    """
    韵律控制器
    
    控制:
    - 语速 (Duration)
    - 音高 (Pitch)
    - 能量 (Energy)
    """
    
    def __init__(self):
        self.speed_scale = 1.0
        self.pitch_scale = 1.0
        self.energy_scale = 1.0
    
    def set_speed(self, scale: float):
        """设置语速 (0.5-2.0)"""
        self.speed_scale = max(0.5, min(2.0, scale))
    
    def set_pitch(self, scale: float):
        """设置音高 (0.5-2.0)"""
        self.pitch_scale = max(0.5, min(2.0, scale))
    
    def set_energy(self, scale: float):
        """设置能量/音量 (0.5-2.0)"""
        self.energy_scale = max(0.5, min(2.0, scale))
    
    def apply_to_mel(self, mel: torch.Tensor) -> torch.Tensor:
        """应用韵律控制到 Mel 频谱"""
        # 语速: 时间轴缩放
        if self.speed_scale != 1.0:
            new_len = int(mel.shape[-1] / self.speed_scale)
            mel = F.interpolate(mel, size=new_len, mode='linear', align_corners=False)
        
        # 能量: 幅度缩放
        mel = mel * self.energy_scale
        
        return mel

prosody = ProsodyController()
prosody.set_speed(1.2)  # 加快 20%
prosody.set_energy(0.8)  # 降低音量

test_mel = torch.randn(1, 80, 100)
modified = prosody.apply_to_mel(test_mel)
print(f"Original: {test_mel.shape} -> Modified: {modified.shape}")

## 3. 说话人嵌入

In [ ]:
class SpeakerEncoder(nn.Module):
    """说话人编码器 (简化版)"""
    
    def __init__(self, n_mels: int = 80, embed_dim: int = 256):
        super().__init__()
        self.lstm = nn.LSTM(n_mels, embed_dim // 2, num_layers=3, 
                           batch_first=True, bidirectional=True)
        self.proj = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        """提取说话人嵌入"""
        # mel: [batch, n_mels, time] -> [batch, time, n_mels]
        mel = mel.transpose(1, 2)
        
        output, _ = self.lstm(mel)
        # 取最后一帧
        embed = output[:, -1, :]
        embed = self.proj(embed)
        embed = F.normalize(embed, dim=-1)
        
        return embed

class MultiSpeakerTTS(nn.Module):
    """多说话人 TTS"""
    
    def __init__(self, tts_model: TextToSpeech, speaker_embed_dim: int = 256):
        super().__init__()
        self.tts = tts_model
        self.speaker_encoder = SpeakerEncoder(embed_dim=speaker_embed_dim)
        
        # 说话人嵌入投影
        self.speaker_proj = nn.Linear(speaker_embed_dim, tts_model.config.encoder_dim)
    
    def forward(self, text: torch.Tensor, reference_mel: torch.Tensor) -> Dict:
        """使用参考音频的说话人风格合成"""
        # 提取说话人嵌入
        speaker_embed = self.speaker_encoder(reference_mel)
        speaker_embed = self.speaker_proj(speaker_embed)
        
        # 编码文本
        encoder_output = self.tts.encoder(text)
        
        # 添加说话人信息
        encoder_output = encoder_output + speaker_embed.unsqueeze(1)
        
        return {'encoder_output': encoder_output, 'speaker_embed': speaker_embed}

# 测试
multi_tts = MultiSpeakerTTS(model)
ref_mel = torch.randn(1, 80, 200)
text = torch.randint(0, 256, (1, 30))

output = multi_tts(text, ref_mel)
print(f"Speaker embed: {output['speaker_embed'].shape}")

## 4. 声码器质量评估

In [ ]:
def compute_mel_cepstral_distortion(pred_mel: torch.Tensor, target_mel: torch.Tensor) -> float:
    """计算 Mel 倒谱失真 (MCD)"""
    # 简化计算
    diff = pred_mel - target_mel
    mcd = torch.sqrt((diff ** 2).mean()) * 10 / torch.log(torch.tensor(10.0))
    return mcd.item()

def compute_f0_rmse(pred_f0: torch.Tensor, target_f0: torch.Tensor) -> float:
    """计算基频 RMSE"""
    # 只计算有效帧
    valid = (target_f0 > 0) & (pred_f0 > 0)
    if valid.sum() == 0:
        return 0.0
    rmse = torch.sqrt(((pred_f0[valid] - target_f0[valid]) ** 2).mean())
    return rmse.item()

# 测试
pred = torch.randn(80, 100)
target = torch.randn(80, 100)
mcd = compute_mel_cepstral_distortion(pred, target)
print(f"MCD: {mcd:.2f} dB")

## 总结

本教程介绍了 TTS 的高级技术：

1. **声学模型训练**: Mel + Postnet + Stop token 损失
2. **韵律控制**: 语速、音高、能量调节
3. **说话人克隆**: 说话人编码器 + 嵌入注入
4. **质量评估**: MCD, F0 RMSE 等指标